# Multi-Agent Financial Advisory System

This notebook implements a comprehensive multi-agent financial advisory system using:
- **LangChain and LangGraph** for agent orchestration
- **OpenRouter API** for LLM capabilities
- **Yahoo Finance** for financial data

## Project Overview
We have built an autonomous Investment Research Agent that demonstrates:
1. **Agent Functions**: Planning, dynamic tool usage, self-reflection, and learning
2. **Workflow Patterns**: Prompt Chaining, Routing, and Evaluator-Optimizer
3. **Multi-Agent Coordination**: Specialized agents for different financial analysis tasks



The entire work on this project can be found on github at : https://github.com/webJdi/finbrAIn



### Importing Required Libraries

In [42]:
try:
    import dotenv
    print("✅ python-dotenv is already installed")
except ImportError:
    print("📦 Installing python-dotenv...")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "python-dotenv"])
    print("✅ python-dotenv installed successfully")

✅ python-dotenv is already installed


In [43]:
import os
import json
from datetime import datetime, timedelta
from typing import Dict, Any, List, TypedDict
import yfinance as yf
from dataclasses import dataclass
import time
from openai import OpenAI
from dotenv import load_dotenv
import requests

### Initializing OpenRouter client

For the LLM inference, we are using openrouter API

In [44]:
load_dotenv()
api_key = os.getenv("Key3")

if not api_key:
    print("Warning: No API key found in .env file!")
    api_key = ""
else:
    print("API key loaded successfully from .env file")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
)

print("Libraries imported successfully!")
print(f" API Key status: {'Loaded' if api_key else 'Missing'}")

API key loaded successfully from .env file
Libraries imported successfully!
 API Key status: Loaded


## 1. Financial Data Tools Implementation

For implementing the tools for fetching financial data from various APIs as specified in the requirements.

In [45]:
class FinancialDataTool:
    """Tool for fetching financial data from Yahoo Finance"""
    
    def __init__(self):
        self.name = "financial_data_tool"
        self.description = "Fetches stock data, financials, and company information"
    
    def get_stock_data(self, symbol: str) -> Dict[str, Any]:
        """Fetch comprehensive stock data"""
        try:
            ticker = yf.Ticker(symbol)
            info = ticker.info
            history = ticker.history(period="1y")
            
            return {
                "symbol": symbol,
                "company_name": info.get("longName", "N/A"),
                "sector": info.get("sector", "N/A"),
                "industry": info.get("industry", "N/A"),
                "current_price": info.get("currentPrice", 0),
                "market_cap": info.get("marketCap", 0),
                "pe_ratio": info.get("trailingPE", 0),
                "forward_pe": info.get("forwardPE", 0),
                "price_to_book": info.get("priceToBook", 0),
                "debt_to_equity": info.get("debtToEquity", 0),
                "profit_margin": info.get("profitMargins", 0),
                "revenue_growth": info.get("revenueGrowth", 0),
                "52_week_high": info.get("fiftyTwoWeekHigh", 0),
                "52_week_low": info.get("fiftyTwoWeekLow", 0),
                "volume": info.get("volume", 0),
                "recommendation": info.get("recommendationKey", "N/A"),
                "analyst_target": info.get("targetMeanPrice", 0),
                "recent_performance": {
                    "1_month": history['Close'].pct_change(21).iloc[-1] if len(history) > 21 else 0,
                    "3_month": history['Close'].pct_change(63).iloc[-1] if len(history) > 63 else 0,
                    "ytd": history['Close'].pct_change().cumsum().iloc[-1] if len(history) > 0 else 0
                },
                "timestamp": datetime.now().isoformat()
            }
        except Exception as e:
            return {"error": str(e), "symbol": symbol}

# Initialize financial tool
financial_tool = FinancialDataTool()

print("Financial data tool initialized!")

Financial data tool initialized!


## Multi-Source Financial Data Integration

Implementation of comprehensive data collection from recommended sources:
- **Yahoo Finance** - Primary financial data and company metrics
- **FRED API** - Economic indicators and macroeconomic data


In [46]:
class EnhancedFinancialDataTool:
    """Enhanced tool using Yahoo Finance and FRED API for comprehensive financial analysis"""
    
    def __init__(self):
        self.name = "enhanced_financial_data_tool"
        self.description = "Comprehensive financial data using Yahoo Finance and FRED API"
        
        # Load FRED API key
        self.fred_api_key = os.getenv("Fred_key")
        if self.fred_api_key:
            print("FRED API key loaded for economic data")
        else:
            print("FRED API key not found - economic data will use fallback")
    
    def get_yahoo_finance_data(self, symbol: str) -> Dict[str, Any]:
        """Yahoo Finance data collection"""
        try:
            ticker = yf.Ticker(symbol)
            info = ticker.info
            history = ticker.history(period="1y")
            
            return {
                "symbol": symbol,
                "company_name": info.get("longName", "N/A"),
                "sector": info.get("sector", "N/A"),
                "industry": info.get("industry", "N/A"),
                "current_price": info.get("currentPrice", 0),
                "market_cap": info.get("marketCap", 0),
                "pe_ratio": info.get("trailingPE", 0),
                "pb_ratio": info.get("priceToBook", 0),
                "dividend_yield": info.get("dividendYield", 0),
                "52_week_high": info.get("fiftyTwoWeekHigh", 0),
                "52_week_low": info.get("fiftyTwoWeekLow", 0),
                "volume": info.get("volume", 0),
                "avg_volume": info.get("averageVolume", 0),
                "beta": info.get("beta", 0),
                "recommendation": info.get("recommendationKey", "N/A"),
                "target_price": info.get("targetMeanPrice", 0),
                "analyst_target": info.get("targetMeanPrice", 0),
                "recent_performance": {
                    "1_month": history['Close'].pct_change(21).iloc[-1] if len(history) > 21 else 0,
                    "3_month": history['Close'].pct_change(63).iloc[-1] if len(history) > 63 else 0,
                    "ytd": history['Close'].pct_change().cumsum().iloc[-1] if len(history) > 0 else 0
                },
                "timestamp": datetime.now().isoformat()
            }
        except Exception as e:
            return {"error": str(e), "symbol": symbol}

    def get_enhanced_yahoo_data(self, symbol: str) -> Dict[str, Any]:
        """To get comprehensive data from Yahoo Finance including additional metrics"""
        try:
            ticker = yf.Ticker(symbol)
            info = ticker.info
            history = ticker.history(period="1y")
            financials = ticker.financials
            balance_sheet = ticker.balance_sheet
            cashflow = ticker.cashflow
            
            # Financial metrics; This can be further improved based on response during fine tuning
            enhanced_data = {
                "symbol": symbol,
                "company_name": info.get("longName", "N/A"),
                "sector": info.get("sector", "N/A"),
                "industry": info.get("industry", "N/A"),
                "current_price": info.get("currentPrice", 0),
                "market_cap": info.get("marketCap", 0),
                "enterprise_value": info.get("enterpriseValue", 0),
                "pe_ratio": info.get("trailingPE", 0),
                "forward_pe": info.get("forwardPE", 0),
                "peg_ratio": info.get("pegRatio", 0),
                "price_to_book": info.get("priceToBook", 0),
                "price_to_sales": info.get("priceToSalesTrailing12Months", 0),
                "debt_to_equity": info.get("debtToEquity", 0),
                "current_ratio": info.get("currentRatio", 0),
                "quick_ratio": info.get("quickRatio", 0),
                "gross_margin": info.get("grossMargins", 0),
                "operating_margin": info.get("operatingMargins", 0),
                "profit_margin": info.get("profitMargins", 0),
                "roe": info.get("returnOnEquity", 0),
                "roa": info.get("returnOnAssets", 0),
                "revenue_growth": info.get("revenueGrowth", 0),
                "earnings_growth": info.get("earningsGrowth", 0),
                "revenue_per_share": info.get("revenuePerShare", 0),
                "book_value": info.get("bookValue", 0),
                "cash_per_share": info.get("totalCashPerShare", 0),
                "free_cash_flow": info.get("freeCashflow", 0),
                "operating_cash_flow": info.get("operatingCashflow", 0),
                "ebitda": info.get("ebitda", 0),
                "total_debt": info.get("totalDebt", 0),
                "total_cash": info.get("totalCash", 0),
                "shares_outstanding": info.get("sharesOutstanding", 0),
                "float_shares": info.get("floatShares", 0),
                "insider_percent": info.get("heldPercentInsiders", 0),
                "institution_percent": info.get("heldPercentInstitutions", 0),
                "short_ratio": info.get("shortRatio", 0),
                "short_percent": info.get("shortPercentOfFloat", 0),
                "timestamp": datetime.now().isoformat()
            }
            
            return enhanced_data
            
        except Exception as e:
            return {"error": str(e), "symbol": symbol, "data_source": "yahoo_finance_enhanced"}

    def get_yahoo_news_data(self, symbol: str) -> Dict[str, Any]:
        """Get Yahoo Finance news data"""
        try:
            ticker = yf.Ticker(symbol)
            news = ticker.news
            
            # Format news data
            formatted_news = []
            for article in news[:10]:  # Latest 10 articles
                formatted_news.append({
                    "title": article.get("title", ""),
                    "link": article.get("link", ""),
                    "published": article.get("providerPublishTime", ""),
                    "publisher": article.get("publisher", ""),
                    "type": article.get("type", "")
                })
            
            return {
                "symbol": symbol,
                "articles": formatted_news,
                "total_articles": len(formatted_news),
                "data_source": "yahoo_finance_news",
                "timestamp": datetime.now().isoformat()
            }
            
        except Exception as e:
            return {"error": str(e), "symbol": symbol, "data_source": "yahoo_finance_news"}

    def get_fred_economic_data(self, indicators: list = None) -> Dict[str, Any]:
        """Get economic data from FRED API"""
        if indicators is None:
            indicators = ["GDP", "UNRATE", "CPIAUCSL", "FEDFUNDS", "UMCSENT"]
        
        if not self.fred_api_key:
            # Adding a Fallback data when API key is not available; This can be done away with
            return {
                "error": "FRED API key not available",
                "fallback_data": {
                    "gdp_growth": {"value": "2.8", "units": "Percent"},
                    "unemployment_rate": {"value": "3.9", "units": "Percent"},
                    "inflation_rate": {"value": "3.2", "units": "Percent"},
                    "federal_funds_rate": {"value": "5.25", "units": "Percent"},
                    "consumer_confidence": {"value": "102.0", "units": "Index"}
                },
                "data_source": "fallback",
                "timestamp": datetime.now().isoformat()
            }
        
        economic_data = {}
        base_url = "https://api.stlouisfed.org/fred/series/observations"
        
        indicator_mapping = {
            "GDP": "gdp_growth",
            "UNRATE": "unemployment_rate", 
            "CPIAUCSL": "inflation_rate",
            "FEDFUNDS": "federal_funds_rate",
            "UMCSENT": "consumer_confidence"
        }
        
        for indicator in indicators:
            try:
                params = {
                    "series_id": indicator,
                    "api_key": self.fred_api_key,
                    "file_type": "json",
                    "limit": 1,
                    "sort_order": "desc"
                }
                
                response = requests.get(base_url, params=params, timeout=10)
                
                if response.status_code == 200:
                    data = response.json()
                    observations = data.get("observations", [])
                    
                    if observations and observations[0].get("value") != ".":
                        value = observations[0]["value"]
                        date = observations[0]["date"]
                        
                        # Determine units based on indicator
                        units = "Percent" if indicator in ["UNRATE", "FEDFUNDS"] else "Index" if indicator == "UMCSENT" else "Percent"
                        
                        economic_data[indicator_mapping.get(indicator, indicator.lower())] = {
                            "value": value,
                            "date": date,
                            "units": units
                        }
                    else:
                        economic_data[indicator_mapping.get(indicator, indicator.lower())] = {
                            "value": "N/A",
                            "date": "N/A", 
                            "units": "N/A"
                        }
                else:
                    economic_data[indicator_mapping.get(indicator, indicator.lower())] = {
                        "error": f"API Error: {response.status_code}",
                        "value": "N/A",
                        "units": "N/A"
                    }
                    
            except Exception as e:
                economic_data[indicator_mapping.get(indicator, indicator.lower())] = {
                    "error": str(e),
                    "value": "N/A",
                    "units": "N/A"
                }
        
        return {
            "economic_indicators": economic_data,
            "data_source": "fred_api",
            "timestamp": datetime.now().isoformat()
        }
    
    def get_comprehensive_data(self, symbol: str) -> Dict[str, Any]:
        """Get comprehensive data from Yahoo Finance and FRED API"""
        print(f"Fetching comprehensive data for {symbol} from Yahoo Finance and FRED API...")
        
        # Yahoo Finance basic data
        print("  Yahoo Finance basic data...")
        yahoo_data = self.get_yahoo_finance_data(symbol)
        
        # Enhanced Yahoo Finance metrics
        print("  Yahoo Finance enhanced metrics...")
        enhanced_yahoo_data = self.get_enhanced_yahoo_data(symbol)
        
        # Yahoo Finance news
        print("  Yahoo Finance news...")
        news_data = self.get_yahoo_news_data(symbol)
        
        # FRED Economic Data
        print("  FRED economic indicators...")
        economic_data = self.get_fred_economic_data()
        
        return {
            "yahoo_finance": yahoo_data,
            "yahoo_enhanced": enhanced_yahoo_data,
            "news_data": news_data,
            "economic_data": economic_data,
            "data_sources_used": ["yahoo_finance", "yahoo_enhanced", "yahoo_news", "fred_api"],
            "timestamp": datetime.now().isoformat()
        }

    def analyze_data(self, comprehensive_data: Dict[str, Any], symbol: str) -> Dict[str, Any]:
        """Analyze comprehensive financial data using Yahoo Finance and FRED data"""
        try:
            analysis = {
                "symbol": symbol,
                "analysis_type": "comprehensive_financial_analysis",
                "data_sources": ["yahoo_finance", "yahoo_enhanced", "yahoo_news", "fred_economic_data"],
                
                # Basic Analysis
                "company_overview": {
                    "name": comprehensive_data["yahoo_finance"].get("company_name", "N/A"),
                    "sector": comprehensive_data["yahoo_finance"].get("sector", "N/A"),
                    "industry": comprehensive_data["yahoo_finance"].get("industry", "N/A"),
                    "current_price": comprehensive_data["yahoo_finance"].get("current_price", 0),
                    "market_cap": comprehensive_data["yahoo_enhanced"].get("market_cap", 0)
                },
                
                # Valuation Metrics
                "valuation_metrics": {
                    "pe_ratio": comprehensive_data["yahoo_finance"].get("pe_ratio", 0),
                    "pb_ratio": comprehensive_data["yahoo_finance"].get("pb_ratio", 0),
                    "forward_pe": comprehensive_data["yahoo_enhanced"].get("forward_pe", 0),
                    "peg_ratio": comprehensive_data["yahoo_enhanced"].get("peg_ratio", 0),
                    "price_to_sales": comprehensive_data["yahoo_enhanced"].get("price_to_sales", 0)
                },
                
                # Financial Health
                "financial_health": {
                    "debt_to_equity": comprehensive_data["yahoo_enhanced"].get("debt_to_equity", 0),
                    "current_ratio": comprehensive_data["yahoo_enhanced"].get("current_ratio", 0),
                    "quick_ratio": comprehensive_data["yahoo_enhanced"].get("quick_ratio", 0),
                    "roe": comprehensive_data["yahoo_enhanced"].get("roe", 0),
                    "roa": comprehensive_data["yahoo_enhanced"].get("roa", 0)
                },
                
                # Profitability
                "profitability": {
                    "gross_margin": comprehensive_data["yahoo_enhanced"].get("gross_margin", 0),
                    "operating_margin": comprehensive_data["yahoo_enhanced"].get("operating_margin", 0),
                    "profit_margin": comprehensive_data["yahoo_enhanced"].get("profit_margin", 0),
                    "revenue_growth": comprehensive_data["yahoo_enhanced"].get("revenue_growth", 0),
                    "earnings_growth": comprehensive_data["yahoo_enhanced"].get("earnings_growth", 0)
                },
                
                # Economic Context
                "economic_context": comprehensive_data.get("economic_data", {}).get("economic_indicators", {}),
                
                # News Analysis
                "news_sentiment": {
                    "total_articles": comprehensive_data["news_data"].get("total_articles", 0),
                    "recent_news_available": comprehensive_data["news_data"].get("total_articles", 0) > 0
                },
                
                "summary": f"Analysis for {symbol}: Current price ${comprehensive_data['yahoo_finance'].get('current_price', 0)}, P/E ratio {comprehensive_data['yahoo_finance'].get('pe_ratio', 0)}, operates in {comprehensive_data['yahoo_finance'].get('sector', 'N/A')} sector. Analysis based on comprehensive Yahoo Finance data, enhanced financial metrics, news sentiment, and FRED economic indicators.",
                
                "timestamp": datetime.now().isoformat()
            }
            
            return analysis
            
        except Exception as e:
            return {
                "error": str(e),
                "symbol": symbol,
                "timestamp": datetime.now().isoformat()
            }

# Initialize enhanced tool
enhanced_financial_tool = EnhancedFinancialDataTool()

FRED API key loaded for economic data


## 2. LLM Integration with OpenRouter

We have created a wrapper for the OpenRouter API to handle LLM interactions for our agents

In [47]:
class LLMAgent:
    """Wrapper for OpenRouter API interactions"""
    
    ### We are using Nvidia Nemotron model here; This can be changed as needed

    def __init__(self, model="nvidia/nemotron-nano-9b-v2:free"):
        self.client = client
        self.model = model
        
    def generate_response(self, messages: List[Dict[str, str]], system_prompt: str = None) -> str:
        """Generate response from LLM"""
        try:
            # Add system prompt if provided
            if system_prompt:
                messages = [{"role": "system", "content": system_prompt}] + messages
            
            completion = self.client.chat.completions.create(
                extra_headers={
                    "HTTP-Referer": "https://finbrain-demo.com", ## Will replace this with vercel link
                    "X-Title": "FinbrAIn Financial Analysis System",
                },
                model=self.model,
                messages=messages,
                temperature=0.1,
                max_tokens=1000
            )
            
            return completion.choices[0].message.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
    
    def analyze_data(self, data: Dict[str, Any], analysis_type: str = "general") -> str:
        """Analyze financial data using LLM"""
        
        system_prompts = {
            "general": "You are a financial analyst. Analyze the provided data and give insights.",
            "fundamental": "You are a fundamental analyst. Focus on company fundamentals, valuation, and long-term prospects.",
            "technical": "You are a technical analyst. Focus on price trends, patterns, and technical indicators.",
            "news": "You are a news analyst. Analyze the sentiment and market impact of news articles.",
            "risk": "You are a risk analyst. Focus on identifying and assessing investment risks."
        }
        
        system_prompt = system_prompts.get(analysis_type, system_prompts["general"])
        
        messages = [
            {
                "role": "user", 
                "content": f"Please analyze this financial data: {json.dumps(data, indent=2)}"
            }
        ]
        
        return self.generate_response(messages, system_prompt)

# Initialize LLM agent
llm_agent = LLMAgent()

# Test the LLM connection
test_response = llm_agent.generate_response([
    {"role": "user", "content": "Hello! Are you ready to help with financial analysis?"}
])

print("LLM Agent Response:")
print(test_response)

LLM Agent Response:
Hello! Yes, I'm ready to assist with financial analysis. 😊 Please share the specific details, questions, or data you'd like help with—whether it's interpreting financial statements, analyzing ratios, budgeting, forecasting, or anything else related to finance. Just let me know!



## 3. Investment Research Agent State and Planning

This is the core Investment Research Agent with state management and research planning capabilities. Building an advanced memory system for the financial analysis agent. The MemoryManager class is designed to help the system learn and improve over time by storing detailed records of each analysis, including quality scores, sector insights, and key lessons learned. Every time an analysis is completed, the system saves not just the results, but also the context, challenges, and suggestions for improvement.

In [48]:
class MemoryManager:
    """Enhanced memory system for learning across runs"""
    
    def __init__(self):
        self.memories = []
        self.performance_history = {}
        self.sector_insights = {}
        self.methodology_improvements = []
        
    def add_analysis_memory(self, symbol: str, sector: str, analysis_result: Dict[str, Any]):
        """Store detailed analysis memory with learning insights"""
        
        memory_entry = {
            "symbol": symbol,
            "sector": sector,
            "timestamp": datetime.now().isoformat(),
            "quality_score": analysis_result.get("quality_score", 0),
            "processing_time": analysis_result.get("processing_time", 0),
            "data_sources_used": analysis_result.get("data_sources", []),
            "workflows_successful": analysis_result.get("workflows_executed", []),
            "key_insights": analysis_result.get("key_insights", ""),
            "challenges_faced": analysis_result.get("challenges", []),
            "improvement_suggestions": analysis_result.get("improvements", [])
        }
        
        self.memories.append(memory_entry)
        
        # For Updating sector insights
        if sector not in self.sector_insights:
            self.sector_insights[sector] = {
                "total_analyses": 0,
                "avg_quality": 0,
                "common_challenges": [],
                "best_practices": []
            }
        
        sector_data = self.sector_insights[sector]
        sector_data["total_analyses"] += 1
        
        # For updating running average
        prev_avg = sector_data["avg_quality"]
        new_count = sector_data["total_analyses"]
        sector_data["avg_quality"] = (prev_avg * (new_count - 1) + memory_entry["quality_score"]) / new_count
        
        # Tracking performance by symbol
        if symbol not in self.performance_history:
            self.performance_history[symbol] = []
        self.performance_history[symbol].append(memory_entry)
        
        # Keeping only last 50 memories to manage size; Change this based on requirements
        mem_lim = 50
        if len(self.memories) > mem_lim:
            self.memories = self.memories[-mem_lim:]

    def get_relevant_memories(self, symbol: str, sector: str = None) -> Dict[str, Any]:
        """For retrieving relevant memories for improving current analysis"""
        
        relevant_memories = {
            "symbol_history": [],
            "sector_insights": {},
            "general_lessons": [],
            "methodology_tips": []
        }
        
        # Get previous analyses for this symbol
        if symbol in self.performance_history:
            relevant_memories["symbol_history"] = self.performance_history[symbol][-3:]  # Last 3 analyses
        
        # Get sector-specific insights
        if sector and sector in self.sector_insights:
            relevant_memories["sector_insights"] = self.sector_insights[sector]
        
        # Get general lessons from recent high-quality analyses
        high_quality_memories = [m for m in self.memories if m.get("quality_score", 0) >= 8.0]
        relevant_memories["general_lessons"] = high_quality_memories[-5:]  # Last 5 high-quality analyses
        
        # Get methodology improvements
        relevant_memories["methodology_tips"] = self.methodology_improvements[-10:]
        
        return relevant_memories
    
    def learn_from_feedback(self, symbol: str, feedback: Dict[str, Any]):
        """Function to handle learning from analysis feedback and update methodology"""
        
        if feedback.get("quality_score", 0) >= 8.0:
            # Extracting successful patterns
            successful_pattern = {
                "timestamp": datetime.now().isoformat(),
                "symbol": symbol,
                "success_factors": feedback.get("success_factors", []),
                "methodology_used": feedback.get("methodology", "standard"),
                "quality_score": feedback["quality_score"]
            }
            self.methodology_improvements.append(successful_pattern)
        
        # Keep only recent improvements
        if len(self.methodology_improvements) > 20:
            self.methodology_improvements = self.methodology_improvements[-20:]
    
    def get_memory_summary(self) -> Dict[str, Any]:
        """Get summary of learning progress"""
        
        if not self.memories:
            return {"status": "No learning data available"}
        
        avg_quality = sum(m.get("quality_score", 0) for m in self.memories) / len(self.memories)
        
        return {
            "total_analyses": len(self.memories),
            "average_quality_score": round(avg_quality, 2),
            "sectors_analyzed": len(self.sector_insights),
            "symbols_tracked": len(self.performance_history),
            "methodology_improvements": len(self.methodology_improvements),
            "recent_trend": "improving" if len(self.memories) >= 5 and 
                          sum(m.get("quality_score", 0) for m in self.memories[-3:]) / 3 > avg_quality 
                          else "stable"
        }

Building an autonomous investment research agent that can plan, collect, analyze, and report on financial data for a given stock symbol, while also learning and improving over time. The AgentState structure keeps track of all the key elements of an analysis session, including the research plan, collected data, results, quality assessment, and memory notes. The InvestmentResearchAgent class uses a language model to dynamically generate research plans, analyze data, assess the quality of its own output, and generate professional reports.

In [49]:
class AgentState(TypedDict):
    """State for the Investment Research Agent (LangGraph-style)"""
    stock_symbol: str
    research_plan: List[str]
    collected_data: Dict[str, Any]
    analysis_results: Dict[str, Any]
    quality_assessment: Dict[str, Any]
    final_report: str
    memory_notes: List[str]
    iteration_count: int
    max_iterations: int

class InvestmentResearchAgent:
    """
    Autonomous Investment Research Agent implementing the requirements:
    - Plans research steps for a given stock symbol
    - Uses tools dynamically (APIs, datasets, retrieval)
    - Self-reflects to assess quality of output
    - Learns across runs (keeps brief memories)
    """
    
    def __init__(self):
        self.llm = llm_agent
        self.financial_tool = enhanced_financial_tool  # Use enhanced tool with Yahoo Finance only
        self.enhanced_tool = enhanced_financial_tool   # Enhanced capabilities with Yahoo Finance
        self.memory_manager = MemoryManager()
        
    def plan_research(self, symbol: str, sector: str = None) -> List[str]:
        """Plan research steps for the given stock symbol using memory-enhanced planning"""
        
        # Get relevant memories to improve planning
        relevant_memories = self.memory_manager.get_relevant_memories(symbol, sector)
        
        memory_context = ""
        if relevant_memories["symbol_history"]:
            memory_context += f"\nPrevious analyses for {symbol}:\n"
            for prev in relevant_memories["symbol_history"]:
                memory_context += f"- Quality: {prev['quality_score']}/10, Key insights: {prev.get('key_insights', 'N/A')}\n"
        
        if relevant_memories["sector_insights"]:
            sector_info = relevant_memories["sector_insights"]
            memory_context += f"\nSector insights: Avg quality {sector_info['avg_quality']:.1f}/10 across {sector_info['total_analyses']} analyses\n"
        
        if relevant_memories["methodology_tips"]:
            memory_context += f"\nSuccessful methodology patterns from recent high-quality analyses:\n"
            for tip in relevant_memories["methodology_tips"][-3:]:
                memory_context += f"- {tip.get('success_factors', ['Standard approach'])}\n"
        
        planning_prompt = f"""You are an expert investment research planner with learning capabilities. 
        Create a comprehensive research plan for stock symbol: {symbol}
        
        {memory_context}
        
        Consider these key areas:
        1. Company fundamentals (financials, ratios, growth)
        2. Market sentiment and news analysis  
        3. Technical analysis and price trends
        4. Industry and sector analysis
        5. Risk assessment and competitive analysis
        
        Based on previous experience and memory insights above, adapt your approach and 
        return a list of 5-7 specific, actionable research steps optimized for this analysis."""
        
        response = self.llm.generate_response([
            {"role": "user", "content": planning_prompt}
        ])
        
        # Parsing the response into a list of steps
        lines = response.split('\n')
        research_steps = []
        for line in lines:
            line = line.strip()
            if line and (line.startswith('-') or line.startswith('*') or any(line.startswith(f'{i}.') for i in range(1, 10))):
                clean_step = line.lstrip('-*0123456789. ').strip()
                if clean_step:
                    research_steps.append(clean_step)
        
        # Fallback if parsing fails; This is only as fallback
        if not research_steps:
            research_steps = [
                "Gather company financial data and key metrics",
                "Analyze recent news and market sentiment",
                "Evaluate competitive positioning and industry trends", 
                "Assess financial health and valuation metrics",
                "Identify key risks and opportunities"
            ]
        
        return research_steps
    
    def collect_data(self, symbol: str) -> Dict[str, Any]:
        """Collect comprehensive data using Yahoo Finance only"""
        
        print(f"Collecting comprehensive data for {symbol}...")
        
        comprehensive_data = self.enhanced_tool.get_comprehensive_data(symbol)
        
        return {
            "comprehensive": comprehensive_data,
            "yahoo_finance": comprehensive_data["yahoo_finance"],
            "yahoo_enhanced": comprehensive_data["yahoo_enhanced"], 
            "news": comprehensive_data["news_data"],
            "data_sources_used": comprehensive_data["data_sources_used"],
            "collection_timestamp": datetime.now().isoformat()
        }
    
    def analyze_data(self, symbol: str, data: Dict[str, Any]) -> Dict[str, Any]:
        """Analyze collected data using Yahoo Finance sources only"""
        
        print(f"Analyzing data for {symbol}...")
        
        # Fundamental analysis using Yahoo Finance basic data
        fundamental_analysis = self.llm.analyze_data(
            data["yahoo_finance"], 
            analysis_type="fundamental"
        )
        
        # Enhanced financial analysis using Yahoo Finance enhanced metrics
        enhanced_analysis = self.llm.analyze_data(
            data["yahoo_enhanced"], 
            analysis_type="enhanced_metrics"
        )
        
        # News sentiment analysis using Yahoo Finance news
        news_analysis = self.llm.analyze_data(
            data["news"], 
            analysis_type="news"
        )
        
        # Risk analysis using comprehensive Yahoo Finance data
        risk_analysis = self.llm.analyze_data(
            {
                "financial_data": data["yahoo_finance"],
                "enhanced_metrics": data["yahoo_enhanced"],
                "news_data": data["news"]
            }, 
            analysis_type="risk"
        )
        
        return {
            "fundamental_analysis": fundamental_analysis,
            "enhanced_financial_analysis": enhanced_analysis,
            "news_sentiment_analysis": news_analysis,
            "risk_analysis": risk_analysis,
            "data_sources_utilized": data.get("data_sources_used", []),
            "analysis_timestamp": datetime.now().isoformat()
        }
    
    def assess_quality(self, analysis: Dict[str, Any]) -> Dict[str, Any]:
        """Self-reflect and assess the quality of analysis"""
        
        quality_prompt = f"""As a quality assessor for investment research, evaluate this analysis:
        
        Analysis: {json.dumps(analysis, indent=2)}
        
        Rate the analysis on a scale of 1-10 for:
        1. Completeness - Are all key aspects covered?
        2. Accuracy - Is the analysis factually sound?
        3. Actionability - Are the insights useful for investment decisions?
        4. Risk Assessment - Are risks properly identified?
        
        Provide an overall quality score and specific feedback on improvements needed.
        Format your response as JSON with scores and feedback."""
        
        quality_response = self.llm.generate_response([
            {"role": "user", "content": quality_prompt}
        ])
        
        # Try to parse as JSON, fallback to text analysis
        try:
            # Look for JSON in the response
            import re
            json_match = re.search(r'\{.*\}', quality_response, re.DOTALL)
            if json_match:
                quality_data = json.loads(json_match.group())
            else:
                quality_data = {"overall_score": 7.0, "feedback": quality_response}
        except:
            quality_data = {"overall_score": 7.0, "feedback": quality_response}
        
        return quality_data
    
    def generate_report(self, symbol: str, research_plan: List[str], analysis: Dict[str, Any]) -> str:
        """Generate final investment research report"""
        
        report_prompt = f"""Generate a comprehensive investment research report for {symbol}.
        
        Research Plan Executed: {research_plan}
        Analysis Results: {json.dumps(analysis, indent=2)}
        
        Structure the report with:
        1. Executive Summary
        2. Company Overview  
        3. Financial Analysis
        4. Market Sentiment & News Analysis
        5. Risk Assessment
        6. Investment Recommendation
        7. Key Takeaways
        
        Make it professional and actionable for investors."""
        
        report = self.llm.generate_response([
            {"role": "user", "content": report_prompt}
        ])
        
        return report
    
    def update_memory(self, symbol: str, sector: str, analysis_result: Dict[str, Any]):
        """Enhanced memory update with structured learning"""
        
        # Extract key learning insights
        learning_insights = {
            "quality_score": analysis_result.get("quality_score", 0),
            "processing_time": analysis_result.get("processing_time", 0),
            "data_sources": analysis_result.get("data_sources_used", []),
            "workflows_executed": analysis_result.get("workflows_executed", []),
            "key_insights": self._extract_key_insights(analysis_result),
            "challenges": self._identify_challenges(analysis_result),
            "improvements": self._suggest_improvements(analysis_result)
        }
        
        # Store in enhanced memory manager
        self.memory_manager.add_analysis_memory(symbol, sector, learning_insights)
        
        # Learn from feedback if high quality
        if learning_insights["quality_score"] >= 7.5:
            feedback = {
                "quality_score": learning_insights["quality_score"],
                "success_factors": [
                    "Comprehensive data collection" if len(learning_insights["data_sources"]) > 2 else None,
                    "Multi-workflow integration" if len(learning_insights["workflows_executed"]) > 2 else None,
                    "Efficient processing" if learning_insights["processing_time"] < 30 else None
                ],
                "methodology": "enhanced" if learning_insights["quality_score"] >= 9.0 else "standard"
            }
            # Remove None values
            feedback["success_factors"] = [f for f in feedback["success_factors"] if f is not None]
            self.memory_manager.learn_from_feedback(symbol, feedback)
    
    def _extract_key_insights(self, analysis_result: Dict[str, Any]) -> str:
        """Extract key insights from analysis for memory storage"""
        final_report = analysis_result.get("final_report", "")
        if len(final_report) > 200:
            # Extract first few sentences as key insights
            sentences = final_report.split('. ')[:3]
            return '. '.join(sentences) + '.' if sentences else "Standard analysis completed"
        return final_report or "Analysis completed successfully"
    
    def _identify_challenges(self, analysis_result: Dict[str, Any]) -> List[str]:
        """Identify challenges faced during analysis"""
        challenges = []
        
        if analysis_result.get("processing_time", 0) > 45:
            challenges.append("Extended processing time")
        
        if analysis_result.get("quality_score", 10) < 7.0:
            challenges.append("Lower quality output")
        
        optimization_result = analysis_result.get("optimization_result", {})
        if optimization_result.get("iterations", 0) >= 3:
            challenges.append("Required multiple optimization iterations")
        
        return challenges or ["No significant challenges identified"]
    
    def _suggest_improvements(self, analysis_result: Dict[str, Any]) -> List[str]:
        """Suggest improvements based on analysis performance"""
        improvements = []
        
        if analysis_result.get("processing_time", 0) > 30:
            improvements.append("Optimize data collection for faster processing")
        
        if analysis_result.get("quality_score", 10) < 8.0:
            improvements.append("Enhance analysis depth and accuracy")
        
        workflows_executed = analysis_result.get("workflows_executed", [])
        if len(workflows_executed) < 3:
            improvements.append("Integrate additional workflow patterns")
        
        return improvements or ["Continue current methodology"]
    
    def get_memory_insights(self) -> Dict[str, Any]:
        """Get memory-based insights for performance tracking"""
        return self.memory_manager.get_memory_summary()

# Initialize the Investment Research Agent with enhanced memory
research_agent = InvestmentResearchAgent()

print("Investment Research Agent initialized!")

Investment Research Agent initialized!


# Workflow Patterns


## 4. Workflow Pattern-1: Prompt Chaining (News Analysis)

Workflow: 

Implementing the News → Preprocess → Classify → Extract → Summarize pipeline

In [50]:
class NewsProcessingChain:
    """
    Implements Prompt Chaining: News → Preprocess → Classify → Extract → Summarize
    """
    
    def __init__(self, llm_agent):
        self.llm = llm_agent
    
    def ingest_news(self, news_data: Dict[str, Any]) -> Dict[str, Any]:
        """Step 1: Ingest and validate news data"""
        
        articles = news_data.get("articles", [])
        valid_articles = []
        
        for article in articles:
            if article.get("title") and article.get("content"):
                valid_articles.append(article)
        
        return {
            "valid_articles": valid_articles,
            "total_articles": len(articles),
            "valid_count": len(valid_articles)
        }
    
    def preprocess_news(self, ingested_data: Dict[str, Any]) -> Dict[str, Any]:
        """Step 2: Preprocess news content"""
        
        articles = ingested_data["valid_articles"]
        
        preprocess_prompt = f"""Clean and preprocess these news articles for analysis:
        {json.dumps(articles, indent=2)}
        
        For each article:
        1. Clean the text and remove noise
        2. Extract key entities (companies, people, financial terms)
        3. Standardize financial terminology
        4. Identify key themes
        
        Return processed articles with extracted entities."""
        
        response = self.llm.generate_response([
            {"role": "user", "content": preprocess_prompt}
        ])
        
        return {
            "preprocessed_articles": articles,  # In production, would parse LLM response
            "preprocessing_notes": response,
            "step": "preprocess"
        }
    
    def classify_news(self, preprocessed_data: Dict[str, Any]) -> Dict[str, Any]:
        """Step 3: Classify news by type and sentiment"""
        
        classify_prompt = f"""Classify these preprocessed news articles:
        {json.dumps(preprocessed_data['preprocessed_articles'], indent=2)}
        
        For each article, classify:
        1. News Type (earnings, M&A, product_launch, regulatory, analyst_rating, etc.)
        2. Sentiment (positive, negative, neutral)
        3. Impact Level (high, medium, low)
        4. Market Relevance (stock_specific, sector_wide, market_wide)
        
        Return classification results with confidence scores."""
        
        response = self.llm.generate_response([
            {"role": "user", "content": classify_prompt}
        ])
        
        return {
            "classified_articles": preprocessed_data["preprocessed_articles"],
            "classifications": response,
            "step": "classify"
        }
    
    def extract_insights(self, classified_data: Dict[str, Any]) -> Dict[str, Any]:
        """Step 4: Extract key insights and information"""
        
        extract_prompt = f"""Extract key insights from these classified articles:
        {json.dumps(classified_data['classified_articles'], indent=2)}
        
        Extract:
        1. Key Financial Figures (revenue, profit, stock price changes)
        2. Important Dates and Events
        3. Investment Implications (bullish/bearish factors)
        4. Risk Factors mentioned
        5. Market Opportunities identified
        
        Focus on actionable investment insights."""
        
        response = self.llm.generate_response([
            {"role": "user", "content": extract_prompt}
        ])
        
        return {
            "extracted_insights": response,
            "articles_processed": len(classified_data["classified_articles"]),
            "step": "extract"
        }
    
    def summarize_analysis(self, extracted_data: Dict[str, Any]) -> str:
        """Step 5: Summarize into final investment insights"""
        
        summarize_prompt = f"""Create a comprehensive investment summary from these insights:
        {extracted_data['extracted_insights']}
        
        Structure the summary with:
        1. Executive Summary - Key takeaways
        2. Investment Thesis - Bull/bear cases
        3. Risk Factors - Key risks identified
        4. Action Items - Recommended next steps
        5. Market Outlook - Overall assessment
        
        Make it concise but comprehensive for investment decision-making."""
        
        summary = self.llm.generate_response([
            {"role": "user", "content": summarize_prompt}
        ])
        
        return summary
    
    def process_news_chain(self, news_data: Dict[str, Any]) -> Dict[str, Any]:
        """Execute the full prompt chaining workflow"""
        
        print("Starting News Processing Chain...")
        
        # Step 1: Ingest
        print(" Ingesting news...")
        ingested = self.ingest_news(news_data)
        
        # Step 2: Preprocess  
        print(" Preprocessing...")
        preprocessed = self.preprocess_news(ingested)
        
        # Step 3: Classify
        print(" Classifying...")
        classified = self.classify_news(preprocessed)
        
        # Step 4: Extract
        print("Extracting insights...")
        extracted = self.extract_insights(classified)
        
        # Step 5: Summarize
        print("Summarizing...")
        summary = self.summarize_analysis(extracted)
        
        print("News Processing Chain completed!")
        
        return {
            "final_summary": summary,
            "chain_steps": {
                "ingested": ingested,
                "preprocessed": preprocessed,
                "classified": classified,
                "extracted": extracted
            },
            "workflow": "prompt_chaining"
        }

# Initialize News Processing Chain
news_chain = NewsProcessingChain(llm_agent)

print("News Processing Chain initialized!")

News Processing Chain initialized!


## 5. Workflow Pattern-2: Routing (Specialist Agents)

Implementing content routing to specialist agents (earnings, news, market analyzers)

In [51]:
class SpecialistRouter:
    """Routes content to appropriate specialist agents"""
    
    def __init__(self, llm_agent):
        self.llm = llm_agent
        self.specialists = {
            "earnings": EarningsAnalyst(llm_agent),
            "news": NewsAnalyst(llm_agent), 
            "market": MarketAnalyst(llm_agent),
            "technical": TechnicalAnalyst(llm_agent)
        }
    
    def route_content(self, content: Dict[str, Any]) -> str:
        """Determine which specialist should handle the content"""
        
        routing_prompt = f"""Analyze this content and determine the best specialist:
        
        Content: {json.dumps(content, indent=2)}
        
        Available specialists:
        - earnings: Financial reports, quarterly results, revenue/profit analysis
        - news: Company news, market sentiment, press releases
        - market: Stock prices, market trends, trading analysis  
        - technical: Chart patterns, technical indicators, price analysis
        
        Return only the specialist name (earnings/news/market/technical)."""
        
        response = self.llm.generate_response([
            {"role": "user", "content": routing_prompt}
        ])
        
        # Parse response to get specialist name
        specialist = response.lower().strip()
        if specialist not in self.specialists:
            specialist = "market"  # Default fallback
        
        return specialist
    
    def process_with_specialist(self, content: Dict[str, Any]) -> Dict[str, Any]:
        """Route content and get specialist analysis"""
        
        specialist_type = self.route_content(content)
        specialist = self.specialists[specialist_type]
        
        analysis = specialist.analyze(content)
        
        return {
            "routed_to": specialist_type,
            "specialist_analysis": analysis,
            "routing_confidence": 0.8  # Would be calculated in production
        }

class EarningsAnalyst:
    """Specialist for earnings and financial analysis"""
    
    def __init__(self, llm_agent):
        self.llm = llm_agent
        self.specialty = "earnings_analysis"
    
    def analyze(self, content: Dict[str, Any]) -> str:
        """Analyze earnings and financial data"""
        
        system_prompt = """You are an expert earnings analyst. Focus on:
        - Revenue growth and trends
        - Profit margins and profitability  
        - Earnings per share analysis
        - Cash flow evaluation
        - Balance sheet strength
        - Forward guidance assessment"""
        
        analysis_prompt = f"""Analyze this financial/earnings content:
        {json.dumps(content, indent=2)}
        
        Provide detailed earnings analysis with specific metrics and insights."""
        
        return self.llm.generate_response([
            {"role": "user", "content": analysis_prompt}
        ], system_prompt)

class NewsAnalyst:
    """Specialist for news and sentiment analysis"""
    
    def __init__(self, llm_agent):
        self.llm = llm_agent
        self.specialty = "news_sentiment"
    
    def analyze(self, content: Dict[str, Any]) -> str:
        """Analyze news sentiment and market impact"""
        
        system_prompt = """You are an expert news analyst. Focus on:
        - Market sentiment analysis
        - News impact assessment
        - Stakeholder reactions
        - Public perception changes
        - Regulatory implications"""
        
        analysis_prompt = f"""Analyze this news content:
        {json.dumps(content, indent=2)}
        
        Provide sentiment analysis and market impact assessment."""
        
        return self.llm.generate_response([
            {"role": "user", "content": analysis_prompt}
        ], system_prompt)

class MarketAnalyst:
    """Specialist for market and trading analysis"""
    
    def __init__(self, llm_agent):
        self.llm = llm_agent
        self.specialty = "market_analysis"
    
    def analyze(self, content: Dict[str, Any]) -> str:
        """Analyze market data and trends"""
        
        system_prompt = """You are an expert market analyst. Focus on:
        - Price action and trends
        - Volume analysis
        - Market breadth
        - Sector rotation
        - Trading patterns"""
        
        analysis_prompt = f"""Analyze this market content:
        {json.dumps(content, indent=2)}
        
        Provide market analysis with trading insights and trend assessment."""
        
        return self.llm.generate_response([
            {"role": "user", "content": analysis_prompt}
        ], system_prompt)

class TechnicalAnalyst:
    """Specialist for technical analysis"""
    
    def __init__(self, llm_agent):
        self.llm = llm_agent
        self.specialty = "technical_analysis"
    
    def analyze(self, content: Dict[str, Any]) -> str:
        """Analyze technical patterns and indicators"""
        
        system_prompt = """You are an expert technical analyst. Focus on:
        - Chart patterns and formations
        - Technical indicators (RSI, MACD, etc.)
        - Support and resistance levels
        - Entry/exit points
        - Risk management levels"""
        
        analysis_prompt = f"""Perform technical analysis on this content:
        {json.dumps(content, indent=2)}
        
        Provide technical analysis with specific levels and trading signals."""
        
        return self.llm.generate_response([
            {"role": "user", "content": analysis_prompt}
        ], system_prompt)

# Initialize the routing system
specialist_router = SpecialistRouter(llm_agent)

print("Specialist Router initialized with all analysts!")

Specialist Router initialized with all analysts!


## 6. Workflow Pattern-3: Evaluator-Optimizer

Workflow:

Implementing the Generate → Evaluate → Refine feedback loop system

In [52]:
class EvaluatorOptimizer:
    """
    For implementing Evaluator-Optimizer pattern: Generate → Evaluate → Refine
    """
    
    def __init__(self, llm_agent, max_iterations=3, quality_threshold=7.5):
        self.llm = llm_agent
        self.max_iterations = max_iterations
        self.quality_threshold = quality_threshold
    
    def evaluate_analysis(self, analysis: str, context: Dict[str, Any] = None) -> Dict[str, Any]:
        """Evaluate the quality of an analysis"""
        
        evaluation_prompt = f"""Evaluate this financial analysis for quality:
        
        Analysis: {analysis}
        Context: {json.dumps(context, indent=2) if context else 'No additional context'}
        
        Rate on a scale of 1-10 for:
        1. Accuracy - Factual correctness (weight: 25%)
        2. Completeness - Coverage of key aspects (weight: 20%)  
        3. Actionability - Useful for investment decisions (weight: 20%)
        4. Clarity - Clear communication (weight: 15%)
        5. Risk Assessment - Proper risk identification (weight: 20%)
        
        Provide:
        - Overall score (1-10)
        - Individual scores for each criterion
        - Specific feedback on weaknesses
        - Suggestions for improvement
        
        Format as JSON."""
        
        response = self.llm.generate_response([
            {"role": "user", "content": evaluation_prompt}
        ])
        
        # Try to parse JSON response
        try:
            import re
            json_match = re.search(r'\{.*\}', response, re.DOTALL)
            if json_match:
                evaluation = json.loads(json_match.group())
            else:
                evaluation = {
                    "overall_score": 4.0,
                    "feedback": response,
                    "suggestions": ["Review factual accuracy", "Enhance completeness"]
                }
        except:
            evaluation = {
                "overall_score": 4.0, 
                "feedback": response,
                "suggestions": ["General improvements needed"]
            }
        
        return evaluation
    
    def optimize_analysis(self, original_analysis: str, evaluation: Dict[str, Any], context: Dict[str, Any] = None) -> str:
        """Optimize analysis based on evaluation feedback"""
        
        optimization_prompt = f"""Improve this financial analysis based on the evaluation feedback:
        
        Original Analysis: {original_analysis}
        
        Evaluation Feedback: {json.dumps(evaluation, indent=2)}
        
        Context: {json.dumps(context, indent=2) if context else 'No additional context'}
        
        Address the specific weaknesses identified and implement the suggestions.
        Maintain the original structure but enhance quality, accuracy, and actionability.
        
        Return the improved analysis."""
        
        improved_analysis = self.llm.generate_response([
            {"role": "user", "content": optimization_prompt}
        ])
        
        return improved_analysis
    
    def iterative_refinement(self, initial_analysis: str, context: Dict[str, Any] = None) -> Dict[str, Any]:
        """Run the full evaluator-optimizer loop"""
        
        current_analysis = initial_analysis
        iteration_history = []
        
        print("Starting Evaluator-Optimizer workflow...")
        
        for iteration in range(self.max_iterations):
            print(f"  Iteration {iteration + 1}/{self.max_iterations}")
            
            # Evaluate current analysis
            evaluation = self.evaluate_analysis(current_analysis, context)
            overall_score = evaluation.get("overall_score", 0)
            
            print(f"Quality Score: {overall_score}/10")
            
            # Record iteration
            iteration_record = {
                "iteration": iteration + 1,
                "analysis": current_analysis,
                "evaluation": evaluation,
                "score": overall_score
            }
            
            # Check if quality threshold is met
            if overall_score >= self.quality_threshold:
                print(f"Quality threshold ({self.quality_threshold}) met!")
                iteration_record["optimization"] = None
                iteration_record["threshold_met"] = True
                iteration_history.append(iteration_record)
                break
            
            # Optimize if not final iteration
            if iteration < self.max_iterations - 1:
                print("Optimizing analysis...")
                optimized_analysis = self.optimize_analysis(current_analysis, evaluation, context)
                current_analysis = optimized_analysis
                iteration_record["optimization"] = optimized_analysis
                iteration_record["threshold_met"] = False
            else:
                print("Max iterations reached")
                iteration_record["optimization"] = None
                iteration_record["threshold_met"] = False
            
            iteration_history.append(iteration_record)
        
        final_evaluation = iteration_history[-1]["evaluation"]
        
        print("Evaluator-Optimizer workflow completed!")
        
        return {
            "final_analysis": current_analysis,
            "final_score": final_evaluation.get("overall_score", 0),
            "iterations": len(iteration_history),
            "threshold_met": final_evaluation.get("overall_score", 0) >= self.quality_threshold,
            "iteration_history": iteration_history,
            "improvement": final_evaluation.get("overall_score", 0) - iteration_history[0]["score"],
            "workflow": "evaluator_optimizer"
        }

# Initialize Evaluator-Optimizer
evaluator_optimizer = EvaluatorOptimizer(llm_agent, max_iterations=3, quality_threshold=8.0)

print("Evaluator-Optimizer system initialized!")

Evaluator-Optimizer system initialized!


## Demonstration of Workflows

### Workflow 1: News Processing Chain (Prompt Chaining)

The News Processing Chain implements a sequential prompt chaining workflow that processes financial news through multiple stages:
1. **Ingest** - Validates and prepares news articles
2. **Preprocess** - Cleans text and extracts entities
3. **Classify** - Categorizes by type, sentiment, and impact
4. **Extract** - Pulls out key insights and financial figures
5. **Summarize** - Creates actionable investment summary

This workflow demonstrates how chaining prompts can progressively refine information into actionable insights.

In [53]:
print("=" * 80)
print("DEMONSTRATION 1: News Processing Chain (Prompt Chaining Workflow)")
print("=" * 80)

# Get real news data for Apple
print("\n1. Fetching news data for AAPL...")
demo_news_data = enhanced_financial_tool.get_yahoo_news_data("AAPL")
print(f"   Retrieved {demo_news_data.get('total_articles', 0)} articles")

## Calling the process news chain function from NewsProcessingChain class to exectute the workflow
chain_result = news_chain.process_news_chain(demo_news_data)

# Display results
print("\n3. Chain Results:")
print(f"   • Total chain steps executed: {len(chain_result.get('chain_steps', {}))}")
print(f"   • Summary length: {len(chain_result.get('final_summary', ''))} characters")
print()

print("4. Final Summary (First 500 characters):")
print("-" * 80)
summary_preview = chain_result.get('final_summary', '')[:500]
print(summary_preview + "..." if len(chain_result.get('final_summary', '')) > 500 else summary_preview)
print("-" * 80)

print("\n5. Chain Step Details:")
ingested = chain_result['chain_steps']['ingested']
print(f"   • Ingestion: {ingested.get('valid_count', 0)}/{ingested.get('total_articles', 0)} valid articles")
print(f"   • Preprocessing: Completed with entity extraction")
print(f"   • Classification: Articles classified by type and sentiment")
print(f"   • Extraction: Key insights and financial figures extracted")
print(f"   • Summarization: Final investment summary generated")

print("\n News Processing Chain demonstration complete!")
print("=" * 80)

DEMONSTRATION 1: News Processing Chain (Prompt Chaining Workflow)

1. Fetching news data for AAPL...
   Retrieved 10 articles
Starting News Processing Chain...
 Ingesting news...
 Preprocessing...
 Classifying...
Extracting insights...
Summarizing...
News Processing Chain completed!

3. Chain Results:
   • Total chain steps executed: 4
   • Summary length: 1705 characters

4. Final Summary (First 500 characters):
--------------------------------------------------------------------------------
**Comprehensive Investment Summary**  

---  
**1. Executive Summary - Key Takeaways**  
*No data available.* The absence of classified articles or financial/domestic data prevents preliminary analysis. No insights on market trends, sector performance, or investment opportunities can be derived.  

---  
**2. Investment Thesis - Bull/Bear Cases**  
*Inconclusive.* Without specific articles or data, a thesis cannot be formulated. A bull case would require positive signals (e.g., strong earnings, ..

### Workflow 2: Specialist Router (Content Routing)

The Specialist Router implements intelligent content routing to specialized analysts:
- **Earnings Analyst** - Financial reports and quarterly results
- **News Analyst** - Market sentiment and press releases
- **Market Analyst** - Stock prices and trading patterns
- **Technical Analyst** - Chart patterns and indicators

The router examines content and automatically directs it to the most appropriate specialist for in-depth analysis.

In [54]:
print("=" * 80)
print("DEMONSTRATION 2: Specialist Router (Content Routing Workflow)")
print("=" * 80)

# Prepare different types of content for routing
print("\n1. Preparing different content types for routing demonstration...")

# Content 1: Financial/Earnings data
financial_content = {
    "type": "financial_data",
    "data": {
        "symbol": "AAPL",
        "revenue": "$394.3B",
        "net_income": "$97.0B",
        "eps": "$6.16",
        "pe_ratio": 34.5,
        "context": "Q4 2024 earnings report with record revenue"
    }
}

# Content 2: News article
news_content = {
    "type": "news_article",
    "data": {
        "title": "Apple Announces New AI Features for iPhone",
        "summary": "Company unveils major AI integration plans",
        "sentiment": "positive",
        "impact": "high"
    }
}

# Content 3: Market data
market_content = {
    "type": "market_data",
    "data": {
        "symbol": "AAPL",
        "current_price": "$252.29",
        "day_change": "+2.3%",
        "volume": "45M shares",
        "52_week_high": "$260.10",
        "52_week_low": "$164.08"
    }
}

print("✓ Financial/Earnings content prepared")
print("✓ News article content prepared")
print("✓ Market data content prepared")

# Route each content type
print("\n2. Routing content to appropriate specialists...")
print()

# Route financial content
print("   a) Financial Content:")
financial_routing = specialist_router.process_with_specialist(financial_content)
print(f" Routed to: {financial_routing['routed_to'].upper()} specialist")
print(f" Confidence: {financial_routing['routing_confidence']}")
print(f" Analysis preview: {financial_routing['specialist_analysis'][:150]}...")
print()

# Route news content
print("   b) News Content:")
news_routing = specialist_router.process_with_specialist(news_content)
print(f"Routed to: {news_routing['routed_to'].upper()} specialist")
print(f"Confidence: {news_routing['routing_confidence']}")
print(f" Analysis preview: {news_routing['specialist_analysis'][:150]}...")
print()

# Route market content
print("   c) Market Content:")
market_routing = specialist_router.process_with_specialist(market_content)
print(f" Routed to: {market_routing['routed_to'].upper()} specialist")
print(f" Confidence: {market_routing['routing_confidence']}")
print(f" Analysis preview: {market_routing['specialist_analysis'][:150]}...")
print()

# Summary
print("\n3. Routing Summary:")
print(f"   • Total content pieces routed: 3")
print(f"   • Specialists utilized: {len(set([financial_routing['routed_to'], news_routing['routed_to'], market_routing['routed_to']]))}")
print(f"   • Available specialists: {len(specialist_router.specialists)}")
print(f"   • Specialist types: {', '.join(specialist_router.specialists.keys())}")

print("\n Specialist Router demonstration complete!")
print("=" * 80)

DEMONSTRATION 2: Specialist Router (Content Routing Workflow)

1. Preparing different content types for routing demonstration...
✓ Financial/Earnings content prepared
✓ News article content prepared
✓ Market data content prepared

2. Routing content to appropriate specialists...

   a) Financial Content:
 Routed to: EARNINGS specialist
 Confidence: 0.8
 Analysis preview: **Detailed Earnings Analysis for Apple (AAPL) - Q4 2024**  
*(Note: Analysis is limited to the provided data; historical context and additional metric...

   b) News Content:
Routed to: NEWS specialist
Confidence: 0.8
 Analysis preview: ### **Market Sentiment Analysis**  
The news article reflects a **strongly positive sentiment** due to Apple's significant investment in AI integratio...

   c) Market Content:
 Routed to: MARKET specialist
 Confidence: 0.8
 Analysis preview: **Market Analysis for AAPL (Apple Inc.)**  

### **1. Price Action and Trends**  
- **Current Price**: AAPL is trading at **$252.29**, up **+2.3% 

### Workflow 3: Evaluator-Optimizer (Quality Refinement)

The Evaluator-Optimizer implements an iterative quality improvement loop:
1. **Generate** - Creates initial analysis
2. **Evaluate** - Assesses quality across multiple criteria (accuracy, completeness, actionability, clarity, risk assessment)
3. **Refine** - Improves analysis based on evaluation feedback
4. **Repeat** - Continues until quality threshold is met or max iterations reached

This workflow ensures high-quality output through systematic refinement and self-improvement.

In [ ]:
print("=" * 80)
print("DEMONSTRATION 3: Evaluator-Optimizer (Quality Refinement Workflow)")
print("=" * 80)

# Passing a sample financial analysis (intentionally basic to show improvement)
print("\n1. Creating initial financial analysis (intentionally basic)...")

initial_analysis = """
Apple Inc. (AAPL) Financial Analysis

Apple is a technology company with strong financials. The stock price is $252.29.
The company has good revenue and profits. The P/E ratio is around 34.5.

Apple makes iPhones and other products. They have a lot of cash. The company is 
doing well in the market. Investors should consider buying the stock.

Risk: Competition exists in the technology sector.
"""

print(" Initial analysis created (deliberately simplified)")
print(f" Analysis length: {len(initial_analysis)} characters")


# Add context for better evaluation
analysis_context = {
    "symbol": "AAPL",
    "sector": "Technology",
    "market_cap": "$3.74T",
    "data_sources": ["Yahoo Finance", "FRED API", "News Data"]
}

# Run the iterative refinement
optimization_result = evaluator_optimizer.iterative_refinement(
    initial_analysis, 
    context=analysis_context
)

# Display results
print("\n2. Optimization Results:")
print(f"   • Initial quality score: {optimization_result['iteration_history'][0]['score']}/10")
print(f"   • Final quality score: {optimization_result['final_score']}/10")
print(f"   • Improvement: +{optimization_result['improvement']:.1f} points")
print(f"   • Iterations completed: {optimization_result['iterations']}")
print(f"   • Quality threshold ({evaluator_optimizer.quality_threshold}) met: {optimization_result['threshold_met']}")

print("\n3. Iteration History:")
for i, iteration in enumerate(optimization_result['iteration_history'], 1):
    print(f"   Iteration {i}:")
    print(f"      - Score: {iteration['score']}/10")
    evaluation = iteration.get('evaluation', {})
    if evaluation.get('feedback'):
        feedback_preview = str(evaluation['feedback'])[:100]
        print(f"      - Feedback: {feedback_preview}...")
    if iteration.get('optimization'):
        print(f"      - Status: Optimized and refined")
    elif iteration.get('threshold_met'):
        print(f"      - Status: Quality threshold achieved!")
    else:
        print(f"      - Status: Max iterations reached")

print("\n5. Final Optimized Analysis (First 600 characters):")
print("-" * 80)
final_preview = optimization_result['final_analysis'][:600]
print(final_preview + "..." if len(optimization_result['final_analysis']) > 600 else final_preview)
print("-" * 80)


DEMONSTRATION 3: Evaluator-Optimizer (Quality Refinement Workflow)

1. Creating initial financial analysis (intentionally basic)...
 Initial analysis created (deliberately simplified)
 Analysis length: 397 characters
Starting Evaluator-Optimizer workflow...
  Iteration 1/3
Quality Score: 7/10
Optimizing analysis...
  Iteration 2/3
Quality Score: 8.5/10
Quality threshold (8.0) met!
Evaluator-Optimizer workflow completed!

2. Optimization Results:
   • Initial quality score: 7/10
   • Final quality score: 8.5/10
   • Improvement: +1.5 points
   • Iterations completed: 2
   • Quality threshold (8.0) met: True

3. Iteration History:
   Iteration 1:
      - Score: 7/10
      - Status: Optimized and refined
   Iteration 2:
      - Score: 8.5/10
      - Status: Quality threshold achieved!

5. Final Optimized Analysis (First 600 characters):
--------------------------------------------------------------------------------
**Improved Financial Analysis for Apple Inc. (AAPL)**  

---

### **Compa

### Workflow Demonstrations Summary

The three workflow patterns have been successfully demonstrated:

1. **News Processing Chain (Prompt Chaining)**
   - Sequential processing through 5 stages
   - Progressive refinement of information
   - Final actionable investment summary

2. **Specialist Router (Content Routing)**
   - Intelligent content classification
   - Automatic routing to specialized analysts
   - Domain-specific expert analysis

3. **Evaluator-Optimizer (Quality Refinement)**
   - Iterative quality improvement loop
   - Multi-criteria evaluation system
   - Measurable quality enhancement

These workflows can be combined in the complete FinbrAIn system to provide comprehensive, high-quality financial analysis.

## 7. Complete Multi-Agent Workflow Integration

The main orchestrator that combines all agents and workflows

In [56]:
class FinbrAInSystem:
    """
    Complete Multi-Agent Financial Advisory System
    Orchestrates all agents and workflows
    """
    
    def __init__(self):
        self.research_agent = research_agent
        self.news_chain = news_chain
        self.specialist_router = specialist_router
        self.evaluator_optimizer = evaluator_optimizer
        self.session_history = []
    
    def comprehensive_analysis(self, symbol: str) -> Dict[str, Any]:
        """
        Run comprehensive financial analysis using all agents and workflows
        """
        
        print(f"Starting comprehensive analysis for {symbol}")
        start_time = time.time()
        
        # Initialize agent state
        state = AgentState(
            stock_symbol=symbol,
            research_plan=[],
            collected_data={},
            analysis_results={},
            quality_assessment={},
            final_report="",
            memory_notes=[],
            iteration_count=0,
            max_iterations=3
        )
        
        try:
            # Step 1: Plan research (with memory-enhanced planning)
            print("Step 1: Planning research (using memory insights)...")
            # Get company sector for better memory retrieval
            temp_data = self.research_agent.enhanced_tool.get_yahoo_finance_data(symbol)
            sector = temp_data.get("sector", "Unknown")
            
            research_plan = self.research_agent.plan_research(symbol, sector)
            state["research_plan"] = research_plan
            print(f"   Research plan: {len(research_plan)} steps")
            print(f"   Sector: {sector}")
            
            # Step 2: Collect data
            print("Step 2: Collecting data...")
            collected_data = self.research_agent.collect_data(symbol)
            
            # Add enhanced Yahoo Finance and FRED economic data
            enhanced_yahoo_data = enhanced_financial_tool.get_enhanced_yahoo_data(symbol)
            fred_economic_data = enhanced_financial_tool.get_fred_economic_data()
            
            # Update collected data with enhanced information
            collected_data["enhanced_yahoo"] = enhanced_yahoo_data
            collected_data["fred_economic"] = fred_economic_data
            
            state["collected_data"] = collected_data
            
            # Step 3: Process news through prompt chaining
            print("Step 3: Processing news (Prompt Chaining)...")
            news_analysis = self.news_chain.process_news_chain(collected_data["news"])
            
            # Step 4: Route different content to specialists
            print("Step 4: Routing to specialists...")
            financial_specialist_analysis = self.specialist_router.process_with_specialist(
                {"type": "financial_data", "data": collected_data["yahoo_finance"]}
            )
            
            news_specialist_analysis = self.specialist_router.process_with_specialist(
                {"type": "news_data", "data": collected_data["news"]}
            )
            
            # Step 5: Analyze data
            print("Step 5: Analyzing data...")
            analysis_results = self.research_agent.analyze_data(symbol, collected_data)
            
            # Combine all analyses - Fix: Use correct keys from analyze_data method
            combined_analysis = {
                "fundamental_analysis": analysis_results["fundamental_analysis"],
                "enhanced_financial_analysis": analysis_results["enhanced_financial_analysis"],
                "news_sentiment": analysis_results["news_sentiment_analysis"],
                "risk_analysis": analysis_results["risk_analysis"],
                "news_chain_summary": news_analysis["final_summary"],
                "specialist_financial": financial_specialist_analysis["specialist_analysis"],
                "specialist_news": news_specialist_analysis["specialist_analysis"],
                "data_sources_utilized": analysis_results.get("data_sources_utilized", [])
            }
            
            state["analysis_results"] = combined_analysis
            
            # Step 6: Generate initial report
            print("Step 6: Generating initial report...")
            initial_report = self.research_agent.generate_report(symbol, research_plan, combined_analysis)
            
            # Step 7: Evaluate and optimize using Evaluator-Optimizer
            print("Step 7: Evaluating and optimizing...")
            optimization_result = self.evaluator_optimizer.iterative_refinement(
                initial_report, 
                context={"symbol": symbol, "data": collected_data}
            )
            
            final_report = optimization_result["final_analysis"]
            state["final_report"] = final_report
            state["quality_assessment"] = {
                "final_score": optimization_result["final_score"],
                "iterations": optimization_result["iterations"],
                "improvement": optimization_result["improvement"],
                "threshold_met": optimization_result["threshold_met"]
            }
            
            # Step 8: Update memory with structured learning
            print("Step 8: Updating agent memory and learning...")
            
            # Prepare comprehensive learning data
            learning_data = {
                "quality_score": optimization_result["final_score"],
                "processing_time": time.time() - start_time,
                "data_sources_used": list(collected_data.keys()),
                "workflows_executed": ["research_planning", "prompt_chaining", "routing", "evaluator_optimizer"],
                "final_report": final_report,
                "optimization_result": optimization_result,
                "threshold_met": optimization_result["threshold_met"],
                "improvement": optimization_result["improvement"]
            }
            
            self.research_agent.update_memory(symbol, sector, learning_data)
            
            # Get memory insights for reporting
            memory_insights = self.research_agent.get_memory_insights()
            
            processing_time = time.time() - start_time
            
            # Create comprehensive result
            result = {
                "success": True,
                "symbol": symbol,
                "research_plan": research_plan,
                "collected_data": collected_data,
                "analysis_results": combined_analysis,
                "news_chain_result": news_analysis,
                "specialist_routing": {
                    "financial": financial_specialist_analysis,
                    "news": news_specialist_analysis
                },
                "optimization_result": optimization_result,
                "final_report": final_report,
                "quality_assessment": state["quality_assessment"],
                "processing_time": processing_time,
                "timestamp": datetime.now().isoformat(),
                "agent_memory_updated": True,
                "memory_insights": memory_insights,
                "learning_progress": {
                    "total_analyses": memory_insights.get("total_analyses", 1),
                    "quality_trend": memory_insights.get("recent_trend", "stable"),
                    "sectors_learned": memory_insights.get("sectors_analyzed", 1)
                },
                "workflows_executed": ["research_planning", "prompt_chaining", "routing", "evaluator_optimizer"]
            }
            
            # Store in session history
            self.session_history.append(result)
            
            print(f"Comprehensive analysis completed in {processing_time:.2f} seconds")
            print(f"Final quality score: {optimization_result['final_score']}/10")
            print(f"Learning progress: {memory_insights.get('total_analyses', 1)} analyses, avg quality {memory_insights.get('average_quality_score', 0):.1f}/10")
            print(f"Quality trend: {memory_insights.get('recent_trend', 'stable').upper()}")
            
            return result
            
        except Exception as e:
            error_result = {
                "success": False,
                "error": str(e),
                "symbol": symbol,
                "timestamp": datetime.now().isoformat()
            }
            return error_result
    
    def get_session_summary(self) -> Dict[str, Any]:
        """Get summary of the current session"""
        
        if not self.session_history:
            return {
                "total_analyses": 0,
                "successful_analyses": 0,
                "symbols_analyzed": [],
                "average_quality_score": 0.0,
                "average_processing_time": 0.0,
                "memory_system": self.research_agent.get_memory_insights(),
                "learning_progress": {
                    "quality_trend": "stable",
                    "sectors_learned": 0,
                    "methodology_improvements": 0
                }
            }
        
        successful_analyses = [result for result in self.session_history if result.get("success", False)]
        symbols = list(set([result["symbol"] for result in self.session_history]))
        
        if successful_analyses:
            avg_quality = sum([result["quality_assessment"]["final_score"] for result in successful_analyses]) / len(successful_analyses)
            avg_processing_time = sum([result["processing_time"] for result in successful_analyses]) / len(successful_analyses)
        else:
            avg_quality = 0.0
            avg_processing_time = 0.0
        
        # Get memory insights
        memory_insights = self.research_agent.get_memory_insights()
        
        return {
            "total_analyses": len(self.session_history),
            "successful_analyses": len(successful_analyses),
            "symbols_analyzed": symbols,
            "average_quality_score": avg_quality,
            "average_processing_time": avg_processing_time,
            "memory_system": memory_insights,
            "learning_progress": {
                "quality_trend": memory_insights.get("recent_trend", "stable"),
                "sectors_learned": memory_insights.get("sectors_analyzed", 0),
                "methodology_improvements": len(successful_analyses)
            }
        }

# Initialize the complete system
finbrain_system = FinbrAInSystem()

## 8. Testing the Complete Agent System

We are testing our multi-agent financial advisory system with a real stock analysis

In [57]:
# Demonstrating Memory-based Learning System
print("\n Memory-based Learning System Demonstration")
print("-" * 50)

# Segment shows current memory state
memory_insights = research_agent.get_memory_insights()
print("Current Memory State:")
print(f" Total analyses remembered: {memory_insights.get('total_analyses', 0)}")
print(f" Average quality score: {memory_insights.get('average_quality_score', 0):.1f}/10")
print(f" Sectors analyzed: {memory_insights.get('sectors_analyzed', 0)}")
print(f" Quality trend: {memory_insights.get('recent_trend', 'No data').upper()}")
print(f" Methodology improvements tracked: {memory_insights.get('methodology_improvements', 0)}")


if memory_insights.get('total_analyses', 0) > 0:
    print(f"\n🎓 Learning Status: ACTIVE ({memory_insights['total_analyses']} analyses completed)")
    print(f" Current Performance: {memory_insights.get('average_quality_score', 0):.1f}/10 average quality")
    print(f"Trend: {memory_insights.get('recent_trend', 'stable').upper()}")
else:
    print("\n🎓 Learning Status: READY (No analyses yet - will learn from first run)")


 Memory-based Learning System Demonstration
--------------------------------------------------
Current Memory State:
 Total analyses remembered: 0
 Average quality score: 0.0/10
 Sectors analyzed: 0
 Quality trend: NO DATA
 Methodology improvements tracked: 0

🎓 Learning Status: READY (No analyses yet - will learn from first run)


In [58]:
# Test the cleaned up comprehensive data collection
print("Testing comprehensive data collection with Yahoo Finance only...")
test_data = enhanced_financial_tool.get_comprehensive_data("AAPL")
print(f"\nData sources collected: {len(test_data)}")
for source, data in test_data.items():
    if isinstance(data, dict) and 'error' not in data:
        print(f"  {source}: SUCCESS - {len(data)} data points")
    else:
        print(f"  {source}: ERROR - {data}")

Testing comprehensive data collection with Yahoo Finance only...
Fetching comprehensive data for AAPL from Yahoo Finance and FRED API...
  Yahoo Finance basic data...
  Yahoo Finance enhanced metrics...
  Yahoo Finance news...
  FRED economic indicators...

Data sources collected: 6
  yahoo_finance: SUCCESS - 19 data points
  yahoo_enhanced: SUCCESS - 37 data points
  news_data: SUCCESS - 5 data points
  economic_data: SUCCESS - 3 data points
  data_sources_used: ERROR - ['yahoo_finance', 'yahoo_enhanced', 'yahoo_news', 'fred_api']
  timestamp: ERROR - 2025-10-19T21:16:51.854851


#### Testing analysis method

In [59]:
analysis_result = enhanced_financial_tool.analyze_data(test_data, "AAPL")
print(f"\nAnalysis completed successfully: {len(analysis_result)} key findings")
print("Analysis summary:", analysis_result.get('summary', 'No summary available')[:200] + "...")


Analysis completed successfully: 11 key findings
Analysis summary: Analysis for AAPL: Current price $252.29, P/E ratio 38.341946, operates in Technology sector. Analysis based on comprehensive Yahoo Finance data, enhanced financial metrics, news sentiment, and FRED e...


## 9. Example Usage and API Integration

Example usage of the three workflows and the multi-agent system, for one chosen stock

In [62]:
chosen_stock ="AAPL"

In [ ]:
# Live Testing of the FinbrAIn System

print("Live Testing: FinbrAIn Multi-Agent Financial Advisory System")
print("=" * 60)


# Test 1: Basic LLM Connection Test
print(" Test 1: LLM Connection Test")
try:
    test_response = llm_agent.generate_response([
        {"role": "user", "content": "Hello! Please confirm you're working and ready for financial analysis."}
    ])
    print("LLM Connection:", "Working" if test_response and len(test_response) > 10 else "Failed")
    print(f"Response: {test_response[:100]}...")
except Exception as e:
    print(f"LLM Connection Failed: {str(e)}")

print()

# Test 2: Individual Workflow Components
print("Test 2: Individual Workflow Components")

# Test Financial Data Tool
print(" Testing Financial Data Tool...")
try:
    financial_data = financial_tool.get_stock_data(chosen_stock)
    print(f" Financial Data: Retrieved {len(financial_data)} fields for AAPL")
    print(f" Current Price: ${financial_data.get('current_price', 'N/A')}")
    print(f" Company: {financial_data.get('company_name', 'N/A')}")
except Exception as e:
    print(f" Financial Data Tool Error: {str(e)}")

# Test Yahoo Finance News
print(" Testing Yahoo Finance News...")
try:
    news_data = enhanced_financial_tool.get_yahoo_news_data(chosen_stock)
    print(f" News Data: Retrieved {len(news_data.get('articles', []))} articles for AAPL")
    print(f" Data Source: {news_data.get('data_source', 'N/A')}")
except Exception as e:
    print(f" Yahoo Finance News Error: {str(e)}")

print()

# Test 3: Prompt Chaining Workflow
print(" Test 3: Prompt Chaining Workflow")
try:
    news_data = enhanced_financial_tool.get_yahoo_news_data(chosen_stock)
    news_result = news_chain.process_news_chain(news_data)
    print(" Prompt Chaining: Completed successfully")
    print(f" Summary Length: {len(news_result['final_summary'])} characters")
    print(f" Chain Steps: {len(news_result['chain_steps'])} steps executed")
except Exception as e:
    print(f" Prompt Chaining Failed: {str(e)}")

print()

# Test 4: Routing Workflow  
print(" Test 4: Routing Workflow")
try:
    financial_data = financial_tool.get_stock_data(chosen_stock)
    routing_result = specialist_router.process_with_specialist({
        "type": "financial_data", 
        "data": financial_data
    })
    print(" Routing Workflow: Completed successfully")
    print(f" Routed to: {routing_result['routed_to']} specialist")
    print(f" Analysis Length: {len(routing_result['specialist_analysis'])} characters")
except Exception as e:
    print(f" Routing Workflow Failed: {str(e)}")

print()

# Test 5: Evaluator-Optimizer Workflow
print(" Test 5: Evaluator-Optimizer Workflow")
try:
    ### This is a manual input; A basic analysis given just to see how the system improves it
    sample_analysis = "Apple Inc. (AAPL) shows strong fundamentals with consistent revenue growth. The stock price has been volatile but maintains upward trend. Recommendation: Buy."
    optimization_result = evaluator_optimizer.iterative_refinement(sample_analysis)
    print(" Evaluator-Optimizer: Completed successfully")
    print(f" Final Score: {optimization_result['final_score']}/10")
    print(f" Iterations: {optimization_result['iterations']}")
    print(f" Improvement: +{optimization_result['improvement']:.1f} points")
except Exception as e:
    print(f" Evaluator-Optimizer Failed: {str(e)}")

print()

# Test 6: Full System Integration Test
print(" Test 6: Full System Integration Test")
try:
    print(f"Starting comprehensive analysis for {chosen_stock}")
    result = finbrain_system.comprehensive_analysis(chosen_stock)
    
    if result["success"]:
        print(" COMPREHENSIVE ANALYSIS COMPLETED SUCCESSFULLY!")
        print(f" Quality Score: {result['quality_assessment']['final_score']}/10")
        print(f" Processing Time: {result['processing_time']:.2f} seconds")
        print(f" Optimization Iterations: {result['quality_assessment']['iterations']}")
        print(f" Quality Improvement: +{result['quality_assessment']['improvement']:.1f}")
        print(f" Threshold Met: {'Yes' if result['quality_assessment']['threshold_met'] else 'No'}")
        
        print("\n Workflow Results:")
        print(f"  • Research Plan: {len(result['research_plan'])} steps")
        print(f"  • Data Sources: {len(result['collected_data'])} types")
        print(f"  • Analysis Components: {len(result['analysis_results'])} analyses")
        print(f"  • Workflows Executed: {len(result['workflows_executed'])}")
        
        print("\n Final Report Preview:")
        report_preview = result['final_report'][:300] + "..." if len(result['final_report']) > 300 else result['final_report']
        print(f"  {report_preview}")
        
    else:
        print(f" Comprehensive Analysis Failed: {result.get('error', 'Unknown error')}")
        
except Exception as e:
    print(f" Full Integration Test Failed: {str(e)}")

print()

# Test Results Summary with Enhanced Memory Insights
session_summary = finbrain_system.get_session_summary()
print(" Test Session Summary with Learning Progress:")
print(f"  • Total Analyses: {session_summary.get('total_analyses', 0)}")
print(f"  • Successful Analyses: {session_summary.get('successful_analyses', 0)}")
print(f"  • Symbols Analyzed: {session_summary.get('symbols_analyzed', [])}")
print(f"  • Average Quality Score: {session_summary.get('average_quality_score', 0):.1f}/10")
print(f"  • Average Processing Time: {session_summary.get('average_processing_time', 0):.1f}s")

memory_system = session_summary.get('memory_system', {})
learning_progress = session_summary.get('learning_progress', {})

print(f"\n Memory System Status:")
print(f"  • Memory Entries: {memory_system.get('total_analyses', 0)}")
print(f"  • Quality Trend: {learning_progress.get('quality_trend', 'stable').upper()}")
print(f"  • Sectors Learned: {learning_progress.get('sectors_learned', 0)}")
print(f"  • Methodology Improvements: {learning_progress.get('methodology_improvements', 0)}")

if memory_system.get('total_analyses', 0) > 1:
    print(f"\n Learning Evidence:")
    print(f"  • System has completed {memory_system['total_analyses']} analyses")
    print(f"  • Average quality: {memory_system.get('average_quality_score', 0):.1f}/10") 
    print(f"  • Learning trend: {learning_progress.get('quality_trend', 'stable')}")
    print("  • LEARNING SYSTEM IS ACTIVE AND FUNCTIONAL")
else:
    print(f"\n Learning Ready:")
    print("  • System ready to learn from first analysis")
    print("  • Will track quality improvements over time")
    print("  • Will accumulate sector-specific insights")
    print("  • LEARNING SYSTEM IS INITIALIZED AND READY")

Live Testing: FinbrAIn Multi-Agent Financial Advisory System
 Test 1: LLM Connection Test
LLM Connection: Working
Response: Hello! Yes, I'm fully operational and ready to assist with financial analysis. Please share the spec...

Test 2: Individual Workflow Components
 Testing Financial Data Tool...
 Financial Data: Retrieved 19 fields for AAPL
 Current Price: $252.29
 Company: Apple Inc.
 Testing Yahoo Finance News...
 News Data: Retrieved 10 articles for AAPL
 Data Source: yahoo_finance_news

 Test 3: Prompt Chaining Workflow
Starting News Processing Chain...
 Ingesting news...
 Preprocessing...
 Classifying...
Extracting insights...
Summarizing...
News Processing Chain completed!
 Prompt Chaining: Completed successfully
 Summary Length: 2109 characters
 Chain Steps: 4 steps executed

 Test 4: Routing Workflow
 Routing Workflow: Completed successfully
 Routed to: market specialist
 Analysis Length: 3237 characters

 Test 5: Evaluator-Optimizer Workflow
Starting Evaluator-Optimizer wor

## Saving the Analysis Report


The analysis report is stored in a separate .txt file and the stock data is saved as a json

In [63]:
reportFilename = f"{chosen_stock}_final_report.txt"
jsonFilename = f"{chosen_stock}_analysis_result.json"

In [64]:
# Save the full final report to a text file for easy reading
if 'result' in locals() and result.get('success', False):
    with open(reportFilename, "w", encoding="utf-8") as f:
        f.write(result['final_report'])
    print(f"Full final report saved to {reportFilename}")
else:
    print("No final report available to save.")

Full final report saved to AAPL_final_report.txt


In [65]:
with open(jsonFilename, "w", encoding="utf-8") as f:
    json.dump(session_summary, f, indent=2)
print(f"Session summary saved to {jsonFilename}")

Session summary saved to AAPL_analysis_result.json
